# Sperner's lemma

Colour the corners of a triangulated triangle so that

1. the three big corners get three different colours, and
2. a point on a side gets one of the colours of that side's corners.

Then some small triangle has all three colours — in fact an odd number of them. This notebook
checks that on random colourings and follows the walk that proves it.

In [2]:
import random

from sperner import find_fully_labeled_cell
from sperner.walk import cells

SIZE = 8  # cuts per side; a grid point is (a, b, c) with a + b + c = SIZE


def random_colouring(size, seed):
    rng = random.Random(seed)
    colours = {}
    for a in range(size + 1):
        for b in range(size + 1 - a):
            point = (a, b, size - a - b)
            allowed = [i for i, v in enumerate(point) if v > 0]  # rules 1 and 2
            colours[point] = rng.choice(allowed)
    return colours


colours = random_colouring(SIZE, seed=7)
full = [cell for cell in cells(3, SIZE) if {colours[p] for p in cell} == {0, 1, 2}]
print(f"{len(full)} of {SIZE**2} small triangles have all three colours")

9 of 64 small triangles have all three colours


## The count is always odd

Try many colourings. The door argument says the number of three-coloured triangles has the
same parity as the number of blue–orange edges on the bottom side, which is odd.

In [4]:
counts = []
for seed in range(200):
    colours = random_colouring(SIZE, seed)
    counts.append(sum({colours[p] for p in cell} == {0, 1, 2} for cell in cells(3, SIZE)))
print("all odd:", all(c % 2 == 1 for c in counts))
print("smallest and largest count:", min(counts), max(counts))

all odd: True
smallest and largest count: 5 23


## The walk

The walk starts at a corner and moves through doors. It asks for a colour only when it
reaches a new point.

In [6]:
colours = random_colouring(SIZE, seed=7)
walk = find_fully_labeled_cell(3, SIZE, lambda p: colours[p], record_path=True)
print("three-coloured triangle:", walk.cell.points, "colours:", walk.cell.labels)
print(f"moves: {walk.pivots}, colours looked at: {walk.labeled} of {len(colours)}")
for step, corners in enumerate(walk.path):
    print(step, corners)

three-coloured triangle: ((1, 6, 1), (0, 7, 1), (0, 6, 2)) colours: (0, 1, 2)
moves: 8, colours looked at: 11 of 45
0 ((8, 0, 0),)
1 ((8, 0, 0), (7, 1, 0))
2 ((7, 1, 0), (6, 2, 0))
3 ((6, 2, 0), (5, 3, 0))
4 ((5, 3, 0), (4, 4, 0))
5 ((4, 4, 0), (3, 5, 0))
6 ((3, 5, 0), (2, 6, 0))
7 ((2, 6, 0), (1, 7, 0))
8 ((2, 6, 0), (1, 7, 0), (1, 6, 1))
9 ((1, 7, 0), (1, 6, 1), (0, 7, 1))
10 ((1, 6, 1), (0, 7, 1), (0, 6, 2))


## On a fine grid the walk sees very little

The same walk on a grid with 1000 cuts per side, for Brouwer's colouring of the map that sends
everything to the point (0.2, 0.5, 0.3).

In [8]:
target = (0.2, 0.5, 0.3)


def label(point):
    size = sum(point)
    return next(i for i, t in enumerate(target) if point[i] > 0 and point[i] / size >= t)


walk = find_fully_labeled_cell(3, 1000, label)
print(walk.cell.points)
print(f"looked at {walk.labeled} of {1001 * 1002 // 2} points")

((200, 499, 301), (199, 500, 301), (199, 499, 302))
looked at 1405 of 501501 points


## Exercises

1. Break rule 2: let the points on the sides take any colour. Find a colouring of a small grid
   with no three-coloured triangle.
2. Prove that the number of blue–orange edges on the bottom side is odd.
3. Count the pairs (small triangle, door of it) in two ways to show that the number of
   three-coloured triangles is odd.
4. Run the walk in four dimensions: `find_fully_labeled_cell(4, 20, label)` with a colouring of
   your own. What must the colouring satisfy?